# Cattle Lameness Detection - Pose Estimation Model Training

This notebook walks through training a YOLOv8-pose model for cattle keypoint detection.

## Steps
1. Dataset preparation (annotated cattle videos/images with keypoints)
2. Data augmentation and preprocessing
3. Model training with YOLOv8-pose
4. Evaluation and metrics
5. Export trained model

In [ ]:
# Install dependencies (run once)
# !pip install ultralytics torch opencv-python matplotlib

In [ ]:
import torch
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Dataset Preparation

You need a dataset of cattle images/video frames annotated with keypoints.

**Keypoints to annotate:**
- Head, neck, spine (front, mid, rear)
- Tail base
- Front legs: left/right (shoulder, knee, hoof)
- Rear legs: left/right (hip, knee, hoof)

**Dataset format:** YOLO pose format (see Ultralytics docs)

```
dataset/
  train/
    images/
    labels/
  val/
    images/
    labels/
  data.yaml
```

In [ ]:
# TODO: Update this path to your dataset config
DATASET_CONFIG = 'path/to/data.yaml'

# Example data.yaml structure:
# path: /path/to/dataset
# train: train/images
# val: val/images
# 
# kpt_shape: [18, 3]  # 18 keypoints, 3 values each (x, y, visibility)
# 
# names:
#   0: cattle

## 2. Model Training

In [ ]:
# Load a pretrained YOLOv8-pose model as starting point
model = YOLO('yolov8n-pose.pt')

# Train on your cattle dataset
# TODO: Adjust hyperparameters as needed
results = model.train(
    data=DATASET_CONFIG,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='runs/pose',
    name='cattle_pose',
)

## 3. Evaluation

In [ ]:
# Validate the model
metrics = model.val()
print(f'mAP50: {metrics.box.map50}')
print(f'mAP50-95: {metrics.box.map}')

## 4. Test on a Sample Video

In [ ]:
# TODO: Replace with path to a test video
TEST_VIDEO = 'path/to/test_video.mp4'

results = model(TEST_VIDEO, stream=True)

for r in results:
    # Access keypoints
    if r.keypoints is not None:
        keypoints = r.keypoints.data  # shape: (num_detections, num_keypoints, 3)
        print(f'Detected {len(keypoints)} cattle with keypoints')
    break  # Just show first frame

## 5. Export Model

Export the trained model for use in the web application backend.

In [ ]:
# Export to the backend ml/models directory
EXPORT_PATH = '../backend/app/ml/models/cattle_pose_best.pt'

# Copy best weights
import shutil
best_weights = 'runs/pose/cattle_pose/weights/best.pt'
shutil.copy(best_weights, EXPORT_PATH)
print(f'Model exported to {EXPORT_PATH}')